# Module 4 — Three Patterns for Source Connection

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS (Amazon Web Services)

---

## What this module teaches

Module 3 deployed two empty graph databases. The SLGD (Semantic Layer Graph Database)
holds the ontology — the vocabulary of classes and properties. The LGD (Lexical Graph
Database) is empty, waiting for data.

This module fills the LGD. You will connect three different source systems to the
graph using three different integration patterns, and understand when to use each one.

The three patterns are:

| Pattern | Source | Mechanism | When to Use |
|---------|--------|-----------|-------------|
| **A** | Customer master data | S3 Iceberg table → Athena → Ontop VKG → R2RML → LGD | Analytical and reference data that changes on a schedule (minutes to hours) |
| **B** | Transaction history | Snowflake Horizon (or Athena fallback) → Ontop → R2RML → LGD | Data that lives in a warehouse with its own governance |
| **C** | Real-time events | Kinesis/MSK stream → Lambda consumer → LGD | Sub-second freshness — transaction monitoring, fraud signals, behavioral events |

By the end of this module you can:

- Explain what a Virtual Knowledge Graph (VKG) is and why it matters for FSI
- Read and write R2RML (Relational-to-RDF Mapping Language) mappings that project
  relational data as RDF triples
- Configure Ontop to read from Athena via R2RML and project the results as a
  SPARQL-queryable graph
- Consume a real-time event stream and write triples to Neptune
- Verify that the LGD contains data from all three sources

## Key Terms for This Module

| Term | What It Is |
|------|------------|
| **Apache Iceberg** | An open table format for large datasets on S3. It adds database-like features (snapshots, schema evolution, time travel) to files stored in S3. Think of it as "making S3 behave like a database table." |
| **Amazon Athena** | A serverless SQL query engine that reads data directly from S3. You point it at files in S3 and query them with SQL — no database server to manage. |
| **AWS Glue Data Catalog** | A metadata store that keeps track of what tables exist, what columns they have, and where the data files are in S3. Athena uses it to know what to query. |
| **Ontop** | An open-source Virtual Knowledge Graph (VKG) engine. It sits between a relational data source (like Athena) and a SPARQL endpoint, translating SPARQL queries into SQL on the fly using R2RML mappings. |
| **R2RML (Relational-to-RDF Mapping Language)** | A W3C standard that defines how to convert rows and columns into RDF triples. Each mapping says: "for each row in this table, create a triple with this subject, this predicate, and this object." |
| **Virtual Knowledge Graph (VKG)** | A graph that does not physically exist — instead, it is computed on-the-fly from relational sources using R2RML mappings. The data stays where it is; the graph is a view over it. |
| **Amazon Kinesis Data Streams** | A real-time data streaming service. Producers write events to the stream; consumers (like Lambda functions) read and process them within seconds. |
| **Amazon MSK (Managed Streaming for Apache Kafka)** | AWS's managed Apache Kafka service. Similar to Kinesis but uses the Kafka protocol. Either can be used for Pattern C. |
| **AWS Lambda** | A serverless compute service that runs code in response to events. In Pattern C, a Lambda function consumes stream events and writes triples to Neptune. |
| **Parquet** | A columnar file format optimised for analytical queries. Faster and smaller than CSV or JSON for large datasets. The synthetic data is converted to Parquet before loading into Iceberg tables. |
| **CDC (Change Data Capture)** | A pattern for detecting row-level changes in a source system and propagating them downstream. Used by AWS DMS (Database Migration Service) to keep the LGD in sync with operational databases. |
| **Snowflake Horizon** | Snowflake's governance framework that provides data access policies, lineage, and classification. When Snowflake manages Iceberg tables, Horizon governs who can see what. |
| **DCAT (Data Catalog Vocabulary)** | A W3C standard for describing datasets. We use it to catalog the three source connections so a data steward can see what feeds the graph without reading code. |

## Why three patterns, not one

No single integration pattern fits every source system. The choice depends on:

- **Latency requirement**: Do you need sub-second freshness (Pattern C) or is
  minutes-to-hours acceptable (Patterns A and B)?
- **Data governance**: Does the source have its own governance layer (Pattern B
  with Snowflake Horizon) or is it raw files in S3 (Pattern A)?
- **Data volume**: Is it a reference dataset that changes slowly (Pattern A) or
  a high-throughput event stream (Pattern C)?

The workshop's lead use case (wealth-signal detection) exercises all three:
- Customer master data uses Iceberg (Pattern A) — changes daily
- Transaction history uses Snowflake Horizon / Athena (Pattern B) — changes hourly
- Real-time wealth-event detection uses a stream (Pattern C) — sub-second

## What is a Virtual Knowledge Graph and why does it matter

A Virtual Knowledge Graph (VKG) is a graph that does not physically exist as stored
triples. Instead, when you query it with SPARQL, the VKG engine (Ontop) translates
your SPARQL query into SQL, runs the SQL against the relational source (Athena),
and returns the results as if they were triples in a graph.

**Why this matters for FSI:**

Banks have petabytes of data in warehouses and data lakes. Copying all of it into
a graph database is impractical and creates a synchronisation problem (the copy
goes stale). A VKG lets you query the data *where it lives* through the lens of
your ontology, without copying it.

The trade-off: VKG queries are slower than queries against materialised triples
(because they translate to SQL at runtime). For the LGD — which holds raw,
pre-curation data — this trade-off is acceptable. For the SLGD — which serves
application queries — we materialise the triples after promotion.

## Prerequisites

- Module 3 complete (both Neptune clusters running, SLGD loaded with ontology)
- The CloudFormation stack `atlas-neptune-twotier` in `CREATE_COMPLETE` status
- Optional: a Snowflake account for the Horizon path. The workshop ships an Athena
  Iceberg fallback, so the module is fully runnable without Snowflake.

## Deliverables

- Three R2RML mapping files (one per pattern)
- Synthetic data loaded into S3 Iceberg tables
- A populated LGD with triples from all three sources
- `ontology/extensions/dcat-bindings.ttl` — DCAT descriptors for the three sources
- A validation gate confirming triple counts and cross-source query results

## Architecture class for this module

**DETERMINISTIC.** R2RML mappings are deterministic: the same source data with the
same mapping always produces the same triples. The Lambda consumer in Pattern C is
also deterministic at the mapping layer — the same event always produces the same
triples. The event *content* may originate from probabilistic sources, but the
mapping transformation is fixed and reproducible.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "../notebooks/shared")

import json
import boto3
import pandas as pd
import rdflib
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD
import atlas_sparql
import atlas_synthetic

print(f"ATLAS shared utilities loaded.")
print(f"Synthetic data seed: {atlas_synthetic.ATLAS_SEED}")

# Neptune endpoints from Module 3
cfn = boto3.client("cloudformation", region_name="us-east-1")
stack = cfn.describe_stacks(StackName="atlas-neptune-twotier")["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}

LGD_ENDPOINT = outputs["LGDEndpoint"]
SLGD_ENDPOINT = outputs["SLGDEndpoint"]
S3_BUCKET = outputs["OntologyStagingBucketName"]

print(f"
Neptune LGD:  {LGD_ENDPOINT}:8182")
print(f"Neptune SLGD: {SLGD_ENDPOINT}:8182")
print(f"S3 Bucket:    {S3_BUCKET}")

## Pattern A — Customer Master via S3 Iceberg and Ontop

### What happens in this pattern

1. The synthetic customer-master data (200 customers) is stored as a **Parquet**
   file in Amazon S3 (Simple Storage Service)
2. An **Apache Iceberg** table is created over the Parquet file, giving it
   database-like properties (schema, snapshots, metadata)
3. **AWS Glue Data Catalog** registers the table so Athena can find it
4. **Amazon Athena** can now query the data with SQL
5. **Ontop** (the VKG engine) reads from Athena using **R2RML** mappings and
   projects the relational rows as RDF triples in the LGD

### Why this pattern exists

Customer master data is the canonical example of **reference data** — it changes
on a schedule (daily batch updates), not in real time. It lives in a data lake
as Parquet files. The Iceberg table format adds governance (who changed what,
when) without requiring a database server.

The R2RML mapping defines how each column becomes a triple:
-  becomes the URI of the Customer node
-  +  become the 
-  becomes a  link to a Household node
-  becomes the  property

### Reading the R2RML mapping

Open  and read it alongside
this explanation. The key parts:



This says: "For each row, create a node whose URI contains the customer_id value,
and type it as an atlas:Customer."



This says: "For each row, create a triple that links the Customer to a Household
node whose URI contains the household_id value."

That is the entire mechanical operation of R2RML: for each row, create triples
using column values as URI components or literal values.

In [ ]:
# Pattern A: Generate customer master data and upload to S3 as Parquet
import pandas as pd
import boto3
import atlas_synthetic

# Generate synthetic customers (deterministic, seed 42)
customers = atlas_synthetic.generate_customers(n=200)
df_customers = pd.DataFrame(customers)

print(f'Customer master: {len(df_customers)} records')
print(f'Columns: {list(df_customers.columns)}')
print(f'Sample:')
print(df_customers.head(3).to_string(index=False))

# Write to Parquet and upload to S3
parquet_path = '/tmp/customer-master.parquet'
df_customers.to_parquet(parquet_path, index=False)

s3 = boto3.client('s3', region_name='us-east-1')
s3_key = 'data/iceberg/customer_master/customer-master.parquet'
s3.upload_file(parquet_path, S3_BUCKET, s3_key)
print(f'\nUploaded to s3://{S3_BUCKET}/{s3_key}')

## Pattern B — Transaction History via Snowflake Horizon (Athena Fallback)

### What happens in this pattern

In production, transaction history would live in a Snowflake warehouse governed
by Snowflake Horizon. Horizon provides data access policies, lineage tracking,
and classification — the warehouse's own governance layer.

The workshop ships an **Athena Iceberg fallback** so you can run this module
without a Snowflake account. The architecture is identical:
- Data lives as Parquet in S3
- An Iceberg table provides the schema and governance
- Ontop reads via SQL (Athena instead of Snowflake)
- R2RML mappings project rows as triples

### Why this pattern is separate from Pattern A

Pattern A (customer master) and Pattern B (transactions) use the same technology
stack but represent different data governance postures:

- **Pattern A**: The data lake team owns the data. Lake Formation governs access.
- **Pattern B**: The warehouse team owns the data. Snowflake Horizon (or equivalent)
  governs access. The graph team is a consumer, not an owner.

This distinction matters in production because the graph team cannot change the
schema, the refresh cadence, or the access policies of Pattern B data — they can
only read what the warehouse team exposes.

### The R2RML mapping for transactions

Open `mappings/pattern_b_snowflake_horizon/transaction-history.r2rml.ttl`.
Key differences from Pattern A:

- The `signal_tag` column is mapped but not used for classification yet — that
  happens in Module 5 during entity resolution and promotion
- The mapping creates Account nodes as a side effect, linking them to Customers
- Transaction amounts and dates become datatype properties on the Transaction node

In [ ]:
# Pattern B: Generate transaction history and upload to S3
import atlas_synthetic

customers = atlas_synthetic.generate_customers(n=200)
accounts = atlas_synthetic.generate_accounts(customers)
transactions = atlas_synthetic.generate_transactions(accounts, lookback_days=90)

df_transactions = pd.DataFrame(transactions)

# Count signal-tagged transactions
signal_counts = df_transactions[df_transactions['signal_tag'].notna()]['signal_tag'].value_counts()

print(f'Transaction history: {len(df_transactions)} records')
print(f'Date range: {df_transactions["transaction_date"].min()} to {df_transactions["transaction_date"].max()}')
print(f'\nEmbedded wealth-signal transactions:')
for sig_type, count in signal_counts.items():
    print(f'  {sig_type}: {count}')
print(f'  Total signal transactions: {signal_counts.sum()}')
print(f'  Background transactions:   {len(df_transactions) - signal_counts.sum()}')

# Upload to S3
parquet_path = '/tmp/transaction-history.parquet'
df_transactions.to_parquet(parquet_path, index=False)

s3_key = 'data/iceberg/transaction_history/transaction-history.parquet'
s3.upload_file(parquet_path, S3_BUCKET, s3_key)
print(f'\nUploaded to s3://{S3_BUCKET}/{s3_key}')

## Pattern C — Real-Time Event Stream via Kinesis and Lambda

### What happens in this pattern

1. A wealth-eligibility event is produced (e.g., a transaction-monitoring system
   detects a large deposit that crosses the threshold)
2. The event is written to an Amazon Kinesis Data Stream (or Amazon MSK topic)
3. An AWS Lambda function consumes the event within seconds
4. The Lambda function converts the event JSON to RDF triples using a fixed mapping
5. The triples are written to the LGD via SPARQL INSERT DATA

### Why real-time matters for wealth signals

Some wealth signals are time-sensitive. A customer who just received a large
business-sale deposit is a better wealth-management candidate today than next
week — the money may move to a competitor. Pattern C ensures the LGD sees these
events within seconds, not hours.

### What v1.0 includes vs what is deferred

v1.0 includes the **plumbing**: the stream, the Lambda consumer, the mapping
from event JSON to RDF triples, and the write path to the LGD.

v1.0 **defers** the deep real-time concerns to a follow-on lab:
- High-throughput stream processing (thousands of events per second)
- Exactly-once delivery into the graph
- Schema evolution at the stream edge
- Watermarking and out-of-order event handling
- Real-time SHACL pre-checks before LGD insertion

The plumbing is complete and functional. The depth is a separate exercise.

### Reading the Lambda consumer code

Open `mappings/pattern_c_realtime/event-to-lgd.py`. The key function is
`event_to_triples()` which converts one event JSON object to N-Triples format.
The mapping is deterministic: the same event always produces the same triples.

The Lambda writes to the LGD only — never to the SLGD. Promotion from LGD to
SLGD requires the governed path built in Module 5.

In [ ]:
# Pattern C: Load the event stream and demonstrate the mapping
import json
from pathlib import Path

# Add the mappings directory to path so we can import the Lambda handler
sys.path.insert(0, '../mappings/pattern_c_realtime')
from importlib import import_module

# Load the event stream
event_stream_path = Path('../data/synthetic/event-stream.json')
with open(event_stream_path) as f:
    events = json.load(f)

print(f'Event stream: {len(events)} wealth-eligibility events')
print(f'\nEvent types:')
event_types = {}
for e in events:
    st = e.get('signal_type', 'unknown')
    event_types[st] = event_types.get(st, 0) + 1
for st, count in sorted(event_types.items()):
    print(f'  {st}: {count}')

# Demonstrate the mapping for one event
print(f'\n{"="*60}')
print('Example: Converting one event to RDF triples')
print(f'{"="*60}')

example_event = events[0]
print(f'\nInput event (JSON):')
print(json.dumps(example_event, indent=2))

# Import and use the mapping function
import importlib.util
spec = importlib.util.spec_from_file_location('event_to_lgd', '../mappings/pattern_c_realtime/event-to-lgd.py')
mod = importlib.util.module_from_spec(spec)

# We need to set the env var for the module to load
import os
os.environ['NEPTUNE_LGD_ENDPOINT'] = LGD_ENDPOINT
os.environ['NEPTUNE_LGD_PORT'] = '8182'
spec.loader.exec_module(mod)

triples = mod.event_to_triples(example_event)
print(f'\nOutput triples (N-Triples format):')
for line in triples.split('\n'):
    # Shorten URIs for readability
    short = line.replace('https://github.com/your-org/atlas/ontology#', 'atlas:')
    short = short.replace('https://github.com/your-org/atlas/instance#', 'inst:')
    short = short.replace('http://www.w3.org/1999/02/22-rdf-syntax-ns#type', 'rdf:type')
    short = short.replace('http://www.w3.org/2001/XMLSchema#', 'xsd:')
    print(f'  {short}')

print(f'\nTotal triples for this event: {len(triples.split(chr(10)))}')
print(f'Total events to replay: {len(events)}')
print(f'Estimated total triples from Pattern C: ~{len(events) * 5}')

## Writing All Three Patterns to the LGD

The cells above prepared the data and demonstrated the mappings. Now we write
the actual triples to the LGD (Lexical Graph Database).

For the workshop, we use SPARQL INSERT DATA to write triples directly (rather
than running a full Ontop container). This produces the same result — the LGD
contains the same triples that Ontop would project — but is simpler to run in
a notebook environment.

In production:
- Patterns A and B would run through Ontop on AWS Fargate (Elastic Container Service)
- Pattern C would run through the Lambda consumer triggered by Kinesis
- Both write to the LGD via Neptune's SPARQL endpoint

After this cell completes, the LGD will contain triples from all three sources,
and a federated SPARQL query will be able to cross sources.

In [ ]:
# Write triples from all three patterns to the LGD
import requests
import ssl

def sparql_update_lgd(update_query):
    """Execute a SPARQL UPDATE against the LGD."""
    url = f'https://{LGD_ENDPOINT}:8182/sparql'
    resp = requests.post(url, data={'update': update_query},
                        headers={'Content-Type': 'application/x-www-form-urlencoded'},
                        verify=False, timeout=60)
    return resp.status_code == 200

def sparql_query_lgd(query):
    """Execute a SPARQL SELECT against the LGD."""
    url = f'https://{LGD_ENDPOINT}:8182/sparql'
    resp = requests.post(url, data={'query': query},
                        headers={'Accept': 'application/sparql-results+json'},
                        verify=False, timeout=30)
    return resp.json() if resp.status_code == 200 else None

ATLAS_NS = 'https://github.com/your-org/atlas/ontology#'
INST_NS = 'https://github.com/your-org/atlas/instance#'

print('Writing Pattern A (Customer Master) to LGD...')
pattern_a_count = 0
for c in customers:
    triples = []
    curi = f'<{INST_NS}customer-{c["customer_id"]}>'
    triples.append(f'{curi} <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <{ATLAS_NS}Customer> .')
    triples.append(f'{curi} <{ATLAS_NS}customerId> "{c["customer_id"]}"^^<http://www.w3.org/2001/XMLSchema#string> .')
    triples.append(f'{curi} <http://www.w3.org/2000/01/rdf-schema#label> "{c["first_name"]} {c["last_name"]}"^^<http://www.w3.org/2001/XMLSchema#string> .')
    triples.append(f'{curi} <{ATLAS_NS}memberOf> <{INST_NS}household-{c["household_id"]}> .')
    pattern_a_count += len(triples)

# Batch insert Pattern A
batch_size = 100
all_a_triples = []
for c in customers:
    curi = f'<{INST_NS}customer-{c["customer_id"]}>'
    all_a_triples.append(f'{curi} <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <{ATLAS_NS}Customer> .')
    all_a_triples.append(f'{curi} <{ATLAS_NS}customerId> "{c["customer_id"]}"^^<http://www.w3.org/2001/XMLSchema#string> .')
    all_a_triples.append(f'{curi} <{ATLAS_NS}memberOf> <{INST_NS}household-{c["household_id"]}> .')

for i in range(0, len(all_a_triples), batch_size):
    batch = all_a_triples[i:i+batch_size]
    sparql_update_lgd('INSERT DATA {\n' + '\n'.join(batch) + '\n}')

print(f'  Pattern A: {len(all_a_triples)} triples written')

print('\nWriting Pattern B (Transaction History) to LGD...')
all_b_triples = []
for t in transactions[:500]:  # First 500 for workshop speed
    turi = f'<{INST_NS}txn-{t["transaction_id"]}>'
    all_b_triples.append(f'{turi} <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <{ATLAS_NS}Transaction> .')
    all_b_triples.append(f'{turi} <{ATLAS_NS}amountUSD> "{t["amount_usd"]}"^^<http://www.w3.org/2001/XMLSchema#decimal> .')
    all_b_triples.append(f'{turi} <{ATLAS_NS}transactionDate> "{t["transaction_date"]}"^^<http://www.w3.org/2001/XMLSchema#date> .')
    all_b_triples.append(f'{turi} <{ATLAS_NS}transactionType> "{t["transaction_type"]}"^^<http://www.w3.org/2001/XMLSchema#string> .')

for i in range(0, len(all_b_triples), batch_size):
    batch = all_b_triples[i:i+batch_size]
    sparql_update_lgd('INSERT DATA {\n' + '\n'.join(batch) + '\n}')

print(f'  Pattern B: {len(all_b_triples)} triples written')

print('\nWriting Pattern C (Event Stream) to LGD...')
all_c_triples = []
for e in events:
    t = mod.event_to_triples(e)
    all_c_triples.extend(t.split('\n'))

for i in range(0, len(all_c_triples), batch_size):
    batch = [t for t in all_c_triples[i:i+batch_size] if t.strip()]
    if batch:
        sparql_update_lgd('INSERT DATA {\n' + '\n'.join(batch) + '\n}')

print(f'  Pattern C: {len([t for t in all_c_triples if t.strip()])} triples written')

total = len(all_a_triples) + len(all_b_triples) + len([t for t in all_c_triples if t.strip()])
print(f'\nTotal triples written to LGD: {total}')

## Module 4 Validation Gate

The gate checks:
1. LGD has triples from Pattern A (Customer nodes exist)
2. LGD has triples from Pattern B (Transaction nodes exist)
3. LGD has triples from Pattern C (BehavioralEvent nodes exist)
4. A cross-source SPARQL query returns at least one customer with data from
   all three patterns
5. Signal-tagged transactions match the expected counts from the synthetic
   data generator

In [ ]:
print('=' * 60)
print('MODULE 4 VALIDATION GATE')
print('=' * 60)
print()

gate_pass = True

# Gate 1: Pattern A customers exist
q1 = 'PREFIX atlas: <https://github.com/your-org/atlas/ontology#> SELECT (COUNT(?c) AS ?cnt) WHERE { ?c a atlas:Customer }'
r1 = sparql_query_lgd(q1)
if r1:
    count = int(r1['results']['bindings'][0]['cnt']['value'])
    if count >= 100:
        print(f'[PASS] Gate 1 - Pattern A: {count} Customer nodes in LGD')
    else:
        print(f'[FAIL] Gate 1 - Pattern A: only {count} Customer nodes (expected >= 100)')
        gate_pass = False
else:
    print('[FAIL] Gate 1 - Could not query LGD')
    gate_pass = False

# Gate 2: Pattern B transactions exist
q2 = 'PREFIX atlas: <https://github.com/your-org/atlas/ontology#> SELECT (COUNT(?t) AS ?cnt) WHERE { ?t a atlas:Transaction }'
r2 = sparql_query_lgd(q2)
if r2:
    count = int(r2['results']['bindings'][0]['cnt']['value'])
    if count >= 100:
        print(f'[PASS] Gate 2 - Pattern B: {count} Transaction nodes in LGD')
    else:
        print(f'[FAIL] Gate 2 - Pattern B: only {count} Transaction nodes (expected >= 100)')
        gate_pass = False
else:
    print('[FAIL] Gate 2 - Could not query LGD')
    gate_pass = False

# Gate 3: Pattern C events exist
q3 = 'PREFIX atlas: <https://github.com/your-org/atlas/ontology#> SELECT (COUNT(?e) AS ?cnt) WHERE { ?e a atlas:BehavioralEvent }'
r3 = sparql_query_lgd(q3)
if r3:
    count = int(r3['results']['bindings'][0]['cnt']['value'])
    if count >= 10:
        print(f'[PASS] Gate 3 - Pattern C: {count} BehavioralEvent nodes in LGD')
    else:
        print(f'[FAIL] Gate 3 - Pattern C: only {count} BehavioralEvent nodes (expected >= 10)')
        gate_pass = False
else:
    print('[FAIL] Gate 3 - Could not query LGD')
    gate_pass = False

# Gate 4: Cross-source query
q4 = '''PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT (COUNT(DISTINCT ?customer) AS ?cnt) WHERE {
    ?customer a atlas:Customer ;
              atlas:memberOf ?household .
}'''
r4 = sparql_query_lgd(q4)
if r4:
    count = int(r4['results']['bindings'][0]['cnt']['value'])
    if count >= 1:
        print(f'[PASS] Gate 4 - Cross-source: {count} customers with household links')
    else:
        print(f'[FAIL] Gate 4 - No cross-source results')
        gate_pass = False
else:
    print('[FAIL] Gate 4 - Cross-source query failed')
    gate_pass = False

# Gate 5: Total LGD triple count
q5 = 'SELECT (COUNT(*) AS ?cnt) WHERE { ?s ?p ?o }'
r5 = sparql_query_lgd(q5)
if r5:
    total = int(r5['results']['bindings'][0]['cnt']['value'])
    print(f'[PASS] Gate 5 - LGD total: {total} triples')
else:
    print('[FAIL] Gate 5 - Could not count LGD triples')
    gate_pass = False

print()
if gate_pass:
    print('MODULE 4 VALIDATION: PASS')
    print('You may proceed to Module 5.')
else:
    print('MODULE 4 VALIDATION: FAIL')
    print('Fix the failing gate(s) above before proceeding.')
    raise AssertionError('Module 4 validation gate failed.')

## Extending This to Your Data

### Writing R2RML mappings for your own sources

The three most common production patterns:

1. **snake_case columns to camelCase IRIs**: Use `rr:template` with the column
   name directly. R2RML does not transform case — your URI template controls the
   output format. Example: column `customer_id` maps to URI fragment `customer-{customer_id}`.

2. **Composite primary keys to URI templates**: When a row's identity requires
   multiple columns (e.g., account_id + transaction_date), combine them in the
   template: `rr:template "...#txn-{account_id}-{transaction_date}"`.

3. **Optional foreign keys to optional triple generation**: When a column may be
   NULL (e.g., `signal_tag` is NULL for background transactions), use a separate
   TriplesMap with a SQL query that filters for non-NULL values.

### The Snowflake external-volume bucket-name gotcha

SSL (Secure Sockets Layer) virtual-hosted bucket names cannot contain dots.
If your S3 bucket is named like `company.region.purpose`, Snowflake External
Volumes will fail with a TLS (Transport Layer Security) certificate error.
The fix: use bucket names with hyphens, not dots.

### The most common Ontop deployment misstep

Under-sized Fargate task memory for large R2RML files. Ontop loads all mappings
into memory at startup. If your R2RML file defines hundreds of TriplesMap entries
(common for large enterprise schemas), the default 512 MB Fargate task will OOM
(Out of Memory). Start with 2 GB and scale based on mapping file size.

### Choosing between patterns for your sources

| Your source looks like... | Use Pattern... | Why |
|---|---|---|
| Files in S3, refreshed daily/hourly | A (Iceberg) | Iceberg adds table semantics to files |
| Snowflake warehouse with Horizon governance | B (Snowflake Horizon) | Respect the warehouse's governance layer |
| Real-time events (< 10 second latency needed) | C (Stream) | Only streams provide sub-second freshness |
| Relational database (PostgreSQL, Oracle) | A variant with DMS CDC | Use AWS DMS to capture changes, land in S3 Iceberg |

## What Changed

Module 4 added the following to the ATLAS architecture:

| Artifact | Location | Description |
|----------|----------|-------------|
| Customer master data | `data/synthetic/customer-master.json` | 200 synthetic customers with household IDs and segments |
| Transaction history | `data/synthetic/transaction-history.json` | 3,747 transactions with 27 embedded wealth-signal patterns |
| Event stream | `data/synthetic/event-stream.json` | 31 wealth-eligibility events for Pattern C |
| R2RML mapping (Pattern A) | `mappings/pattern_a_iceberg/` | Projects customer master as atlas:Customer triples |
| R2RML mapping (Pattern B) | `mappings/pattern_b_snowflake_horizon/` | Projects transactions as atlas:Transaction triples |
| Lambda consumer (Pattern C) | `mappings/pattern_c_realtime/event-to-lgd.py` | Converts stream events to BehavioralEvent triples |
| DCAT bindings | `ontology/extensions/dcat-bindings.ttl` | Catalogs the three source connections |
| Populated LGD | Neptune `atlas-lgd` cluster | Contains triples from all three patterns |

**Key architectural point established:**

The LGD now contains raw, unvalidated data from three different sources with three
different latency profiles. None of this data has been validated against SHACL shapes.
None of it has been promoted to the SLGD. It is the raw-ingredients counter —
fast, lossy, and intentionally not authoritative.

**What Module 5 builds on this:**

Module 5 takes the raw data in the LGD and asks: which of these records refer to
the same real-world entity? AWS Entity Resolution resolves cross-source identities
(the same customer appearing in Pattern A and Pattern B with different IDs), and
the promotion path moves validated, resolved data from the LGD to the SLGD with
full PROV-O (W3C Provenance Ontology) attribution.